In [376]:
import json
import pandas as pd
import datetime
import csv
import os

In [377]:
#Grabs TMDB API key from .txt file
with open('tmdbkey.txt', 'r') as f:
    tmdbkey = f.read()


In [378]:
#which movie should we search

movierequest = input()

In [379]:
#inputs api key into TMDB API
from tmdbv3api import TMDb
tmdb = TMDb()
tmdb.api_key = tmdbkey



In [380]:
#defines the search for the movie module
from tmdbv3api import Movie

movie = Movie()


In [381]:
#Returns the top result in TMDB search for movie title
i = 0
search = movie.search(movierequest)

for res in search:
    
    if i == 0 :
        res_id = res.id
    i += 1


In [382]:
m = movie.details(res_id)


In [383]:
#Pulls release dates, needs additional work to get official release date
releaseday = m.release_date
releaseyear = m.release_date[0:4]
title = m.title

In [384]:
#Pulls Genre details, can be one or more
genres = m.genres

genre = []

for name in genres:
    genre.append (name["name"])

In [385]:
#Pulls Actor name and gender data

cast = m.casts.cast
d = []
for name in cast:
    if name["gender"] == 1:
        Gender = 'Female'
    elif name["gender"] == 2 :
        Gender = 'Male'
    elif name["gender"] > 2:
        Gender = 'Non-Binary'
    else :
        gender = ''

    
    d.append(
        {
            'Name': name["name"],
            'Gender':  Gender
        }
    )

    
actorsdf = pd.DataFrame(d)

In [386]:
#Pulls film crew data into a dataframe to be organized by job
crew = m.casts.crew
e = []
for name in crew:
    if name["gender"] == 1:
        Gender = 'Female'
    elif name["gender"] == 2 :
        Gender = 'Male'
    elif name["gender"] > 2:
        Gender = 'Non-Binary'
    else :
        gender = ''

    
    e.append(
        {
            'Name': name["name"],
            'Job' : name["job"],
            'Gender': Gender
        }
    )

    
crewdf = pd.DataFrame(e)

In [387]:
#Pulling the people we need to record

#Pulls any crew with job type Director, as movies can have one or more directors
directors = crewdf[crewdf["Job"] == "Director"]["Name"].to_list()

#Pulls top billed actor as lead
lead_actor = actorsdf.iloc[0, 0]

#Supporting actors are anyone but the lead
supporting_actors = actorsdf.drop([0])

#Only taking top five other billed actors as support, too many names otherwise
top5support = supporting_actors.head()
supportingactorslist = top5support["Name"].to_list()

#Converting lists to strings for better text formatting in csv
string_list = [str(element) for element in supportingactorslist]
delimiter = ", "
supporting = delimiter.join(string_list)

string_list = [str(element) for element in directors]
direct = delimiter.join(string_list)

string_list = [str(element) for element in genre]
gen = delimiter.join(string_list)


In [388]:
#Checks if there is a csv named 'Movies.csv' in directory already, if not, makes the csv
if os.path.exists('Movies.csv') == False:
    with open('movies.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        field = ["Title", "Director(s)", "Lead Actor", "Top 5 supporting actors", "Genre(s)", "Release Day", "Release Year"]
        writer.writerow(field)
    


In [389]:
#Appends the movie details to the file
with open('movies.csv', 'a', newline='') as file:
    writer = csv.writer(file)      
    newmovie = [title, direct, lead_actor, supporting, gen, releaseday, releaseyear]
    writer.writerow(newmovie)
